In [23]:
import pandas as pd
import numpy as np

In [24]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\2025_Dr. Karni Singh Shooting Range, Delhi - DPCC.xlsx",skiprows=16)

In [25]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,NH3,SO2,CO,Ozone,Benzene,Toluene,RH,WS,WD,BP,AT,RF,TOT-RF
0,01-01-2025 00:00,02-01-2025 00:00,184.64,254.39,24.66,64.29,52.24,31.91,15.56,1.47,23.09,0.16,0.99,84.88,0.99,240.85,991.04,11.27,0.0,0.0
1,02-01-2025 00:00,03-01-2025 00:00,207.12,286.58,29.56,65.88,57.07,51.28,17.60,1.90,23.18,0.22,1.41,84.75,0.99,246.24,990.31,11.46,0.0,0.0
2,03-01-2025 00:00,04-01-2025 00:00,295.92,406.67,59.31,86.64,92.32,61.66,26.80,3.32,26.22,0.41,3.45,84.30,0.57,204.06,990.14,12.22,0.0,0.0
3,04-01-2025 00:00,05-01-2025 00:00,272.03,336.54,57.62,63.98,78.87,62.33,19.31,3.21,20.85,0.46,5.60,89.55,0.98,189.97,988.56,11.77,0.0,0.0
4,05-01-2025 00:00,06-01-2025 00:00,164.17,214.46,12.28,50.58,34.89,47.40,13.27,1.11,20.78,0.22,0.76,88.25,1.14,199.73,988.40,11.36,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,12-11-2025 00:00,13-11-2025 00:00,333.25,491.00,21.04,85.98,62.81,36.31,15.44,1.44,37.57,0.55,1.35,58.39,0.81,150.17,991.71,18.00,0.0,0.0
316,13-11-2025 00:00,14-11-2025 00:00,304.75,463.62,24.12,83.63,64.10,38.89,24.32,1.57,47.99,0.47,1.00,57.47,0.89,99.83,993.17,18.17,0.0,0.0
317,14-11-2025 00:00,15-11-2025 00:00,221.17,366.50,20.20,82.43,60.26,34.68,23.68,1.33,46.32,0.38,0.61,55.62,0.94,117.17,993.04,18.08,0.0,0.0
318,15-11-2025 00:00,16-11-2025 00:00,248.33,390.08,22.21,84.91,63.23,35.38,21.98,1.39,43.45,0.39,0.64,56.52,0.78,176.80,994.05,17.90,0.0,0.0


In [26]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (320, 20)


In [27]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): []
Dropped rows (>70% NaN): 1
Missing values after imputation:
 From Date    0
To Date      0
PM2.5        0
PM10         0
NO           0
NO2          0
NOx          0
NH3          0
SO2          0
CO           0
Ozone        0
Benzene      0
Toluene      0
RH           0
WS           0
WD           0
BP           0
AT           0
RF           0
TOT-RF       0
dtype: int64


In [28]:

# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")

# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")

In [ ]:
# ---------- 5. Convert date columns to datetime ----------
if 'From Date' in df.columns:
    df['From Date'] = pd.to_datetime(df['From Date'], errors='coerce')
if 'To Date' in df.columns:
    df['To Date'] = pd.to_datetime(df['To Date'], errors='coerce')

In [29]:
# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (319, 20)
          From Date           To Date   PM2.5    PM10     NO    NO2    NOx  \
0  01-01-2025 00:00  02-01-2025 00:00  184.64  254.39  24.66  64.29  52.24   
1  02-01-2025 00:00  03-01-2025 00:00   53.32  286.58  29.56  65.88  57.07   
2  03-01-2025 00:00  04-01-2025 00:00   53.32  406.67  13.14  86.64  33.05   
3  04-01-2025 00:00  05-01-2025 00:00   53.32  336.54  13.14  63.98  33.05   
4  05-01-2025 00:00  06-01-2025 00:00  164.17  214.46  12.28  50.58  34.89   

     NH3    SO2    CO  Ozone  Benzene  Toluene     RH    WS      WD      BP  \
0  31.91  15.56  1.47  23.09     0.16     0.99  84.88  0.99  240.85  991.04   
1  28.18  17.60  1.03  23.18     0.22     1.41  84.75  0.99  246.24  990.31   
2  28.18  26.80  1.03  26.22     0.41     3.45  84.30  0.57  204.06  990.14   
3  28.18  19.31  1.03  20.85     0.46     5.60  89.55  0.98  189.97  988.56   
4  47.40  13.27  1.11  20.78     0.22     0.76  88.25  1.14  199.73  988.40   

      AT   RF  TOT-RF  
0  11.27 

In [30]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [31]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,NH3,SO2,CO,Ozone,Benzene,Toluene,RH,WS,WD,BP,AT,RF,TOT-RF
0,01-01-2025 00:00,02-01-2025 00:00,3.036845,1.213415,1.286810,1.023598,1.116650,0.442316,0.395283,1.852611,-1.108058,-0.680957,-0.693844,1.299710,-0.326857,1.186967,1.652643,-2.294160,0.0,0.0
1,02-01-2025 00:00,03-01-2025 00:00,-0.191291,1.605737,1.922762,1.113861,1.449048,-0.074154,0.685239,0.056807,-1.102724,-0.590541,-0.573956,1.291150,-0.326857,1.343197,1.517678,-2.261720,0.0,0.0
2,03-01-2025 00:00,04-01-2025 00:00,-0.191291,3.069360,-0.208325,2.292393,-0.203995,-0.074154,1.992884,0.056807,-0.922531,-0.304222,0.008358,1.261520,-1.489073,0.120605,1.486248,-2.131963,0.0,0.0
3,04-01-2025 00:00,05-01-2025 00:00,-0.191291,2.214636,-0.208325,1.005999,-0.203995,-0.074154,0.928291,0.056807,-1.240832,-0.228875,0.622071,1.607208,-0.354529,-0.287795,1.194134,-2.208793,0.0,0.0
4,05-01-2025 00:00,06-01-2025 00:00,2.533647,0.726759,-0.319941,0.245290,-0.077367,2.587123,0.069793,0.383317,-1.244981,-0.590541,-0.759498,1.521609,0.088220,-0.004900,1.164552,-2.278794,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
314,12-11-2025 00:00,13-11-2025 00:00,-0.191291,-0.091403,0.816985,2.254926,1.844071,1.051558,0.378227,1.730170,-0.249772,-0.093251,-0.591083,-0.444535,-0.824949,-1.441402,1.776514,-1.145126,0.0,0.0
315,13-11-2025 00:00,14-11-2025 00:00,-0.191291,-0.091403,1.216726,2.121518,1.932848,1.408795,1.640388,2.260749,0.367863,-0.213806,-0.690990,-0.505113,-0.603575,-0.067508,2.046443,-1.116102,0.0,0.0
316,14-11-2025 00:00,15-11-2025 00:00,-0.191291,2.579780,0.707964,2.053394,1.668582,0.825861,1.549422,1.281219,0.268875,-0.349431,-0.802315,-0.626927,-0.465216,-2.397910,2.022408,-1.131468,0.0,0.0
317,15-11-2025 00:00,16-11-2025 00:00,-0.191291,2.867166,0.968834,2.194182,1.872975,0.922786,1.307792,1.526101,0.098759,-0.334361,-0.793751,-0.567666,-0.907965,-0.669529,2.209140,-1.162200,0.0,0.0


In [32]:
df.to_excel('DRKarni2025.xlsx', index=False)